# 00. Pedestrian networks from OpenStreetMap

This notebook builds the walking network of each of the 92 metropolitan areas from
an offline OpenStreetMap extract of Mexico. No Overpass server is queried, so the
networks depend only on the extract and can be rebuilt from it.

The national extract from Geofabrik is reduced in two passes with osmium. The
first keeps ways tagged as highways. The second removes the ways that the OSMnx
walking network excludes: motorways and their links, cycleways, bus guideways,
platforms, raceways, ways under construction, proposed or abandoned, areas, ways
where walking is not allowed, and private service and access ways. Streets inside
gated developments are tagged as private, so they are absent from the networks by
construction.

For each area, the municipalities that compose it are dissolved into one polygon
in the local UTM zone. Slivers between municipalities are closed with a 50 m
buffer and its reverse, and vertices are simplified at 25 m. The walk layer is cut
locally with osmium to that polygon plus a 1 km margin, built into a graph with
OSMnx, truncated to the polygon itself, simplified and projected to UTM.

Every connected component of the network is kept. A metropolitan area is a union
of whole municipalities and often holds settlements that are not joined to the
main built-up area by any walkable way: a detached town, an island, a
neighbourhood whose only link is a road the walking filter removes. Keeping their
streets means a block there is reached only by destinations in its own part of the
network, and a block with no destination in reach receives no distance rather than
an arbitrary one.

Outputs, under `Networks/pedestrian_networks/`: one file per area,
`metro_<name>_pedestrian.gpkg`, holding the undirected edges with their length and
geometry, and `_extraction_log.csv`, one row per area with its edge count, its
component structure and its runtime. The walk layer is kept as
`Inputs/osm/mexico_walk.osm.pbf`, with the extract's snapshot date recorded beside
it.

Runtime is about a minute per area and longer for the three largest. A High-RAM
runtime is safer for Mexico City, Monterrey and Guadalajara. Areas already written
are skipped, so an interrupted run resumes where it stopped.

## 1. Setup

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y osmium-tool > /dev/null
!pip -q install osmnx pyogrio
!osmium --version | head -1

In [ ]:
import os, sys, json, time, shutil, subprocess, traceback, warnings, importlib

if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

PROJECT = os.environ.get('MXO_BASE', '/content/drive/MyDrive/Mexico_Obesity')
os.environ['MXO_BASE'] = PROJECT
sys.path.insert(0, f'{PROJECT}/Code')
import mxo_utils as mx
importlib.reload(mx)
assert mx.BASE == PROJECT, f'the module resolves to {mx.BASE}'

import pandas as pd
import geopandas as gpd
import osmnx as ox
from shapely.geometry import Polygon, mapping
from shapely.ops import unary_union

warnings.filterwarnings('ignore')
print('OSMnx', ox.__version__)

INPUT_DIR = f'{mx.DIRS["inputs"]}/metro_areas'
OUTPUT_DIR = mx.DIRS['networks']
OSM_DIR = f'{mx.DIRS["inputs"]}/osm'
WORK_DIR = '/content/osm_work'
LOG_FILE = f'{OUTPUT_DIR}/_extraction_log.csv'

MIRRORS = [
    'https://download.geofabrik.de/north-america/mexico-latest.osm.pbf',
    'https://download.openstreetmap.fr/extracts/north-america/mexico-latest.osm.pbf',
]
RAW_PBF = f'{WORK_DIR}/mexico-latest.osm.pbf'
RAW_PBF_DRIVE = f'{OSM_DIR}/mexico-latest.osm.pbf'
HWY_PBF = f'{WORK_DIR}/mexico_highways.osm.pbf'
WALK_PBF = f'{WORK_DIR}/mexico_walk.osm.pbf'
WALK_PBF_DRIVE = f'{OSM_DIR}/mexico_walk.osm.pbf'

for d in (OUTPUT_DIR, OSM_DIR, WORK_DIR):
    os.makedirs(d, exist_ok=True)

CLOSE_GAPS_M = 50          # closes slivers between municipality polygons
SIMPLIFY_M = 25            # vertex simplification tolerance
EXTRACT_BUFFER_M = 1000    # cut slightly larger, then truncate to the boundary

ox.settings.use_cache = False
ox.settings.log_console = False

# The ways the OSMnx walking network excludes.
WALK_EXCLUDE = [
    'w/highway=abandoned,bus_guideway,construction,cycleway,motorway,'
    'motorway_link,no,planned,platform,proposed,raceway,razed',
    'w/area=yes',
    'w/foot=no',
    'w/service=private',
    'w/access=private',
]
print('boundaries from', INPUT_DIR)
print('writing to     ', OUTPUT_DIR)

## 2. The walk layer

The national extract is used from `Inputs/osm/` if a copy is there, and is
downloaded otherwise. Its snapshot date, the time of the last OpenStreetMap edit
it contains, is read from the file header and written beside the walk layer, since
that is the date the networks describe.

In [ ]:
def run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(' '.join(cmd) + '\n' + r.stderr.strip())


def get_raw_pbf():
    if os.path.exists(RAW_PBF_DRIVE):
        shutil.copy(RAW_PBF_DRIVE, RAW_PBF)
        return 'copy in Inputs/osm'
    part = RAW_PBF + '.part'
    for url in MIRRORS:
        print(f'downloading {url}')
        r = subprocess.run(['wget', '-4', '-q', '--tries=3', '--timeout=60', '-O', part, url])
        if r.returncode == 0 and os.path.getsize(part) > 100e6:
            os.replace(part, RAW_PBF)
            return url
    raise RuntimeError('no mirror reachable; place mexico-latest.osm.pbf in ' + OSM_DIR)


def snapshot_date(pbf):
    r = subprocess.run(['osmium', 'fileinfo', '-g',
                        'header.option.osmosis_replication_timestamp', pbf],
                       capture_output=True, text=True)
    return r.stdout.strip() or 'unknown'


if os.path.exists(WALK_PBF_DRIVE):
    shutil.copy(WALK_PBF_DRIVE, WALK_PBF)
    print('walk layer read from', WALK_PBF_DRIVE)
else:
    source = get_raw_pbf()
    stamp = snapshot_date(RAW_PBF)
    run(['osmium', 'tags-filter', RAW_PBF, 'w/highway', '-o', HWY_PBF, '--overwrite'])
    run(['osmium', 'tags-filter', '--invert-match', HWY_PBF, *WALK_EXCLUDE,
         '-o', WALK_PBF, '--overwrite'])
    os.remove(HWY_PBF)
    shutil.copy(WALK_PBF, WALK_PBF_DRIVE)
    with open(WALK_PBF_DRIVE + '.source.txt', 'w') as f:
        f.write(f'source: {source}\nosm snapshot: {stamp}\nprocessed: {time.ctime()}\n')

meta = WALK_PBF_DRIVE + '.source.txt'
if os.path.exists(meta):
    print(open(meta).read())
print(f'walk layer: {os.path.getsize(WALK_PBF) / 1e6:.0f} MB')

## 3. Building each network

In [ ]:
def metro_polygons(path):
    """The dissolved area polygon and the polygon with its margin, in degrees."""
    gdf = gpd.read_file(path)
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty]
    if gdf.crs is None:
        gdf = gdf.set_crs(4326)
    utm = gdf.estimate_utm_crs()
    poly = unary_union(gdf.to_crs(utm).geometry.buffer(0).tolist())
    poly = poly.buffer(CLOSE_GAPS_M).buffer(-CLOSE_GAPS_M)
    poly = poly.simplify(SIMPLIFY_M, preserve_topology=True)
    buf = poly.buffer(EXTRACT_BUFFER_M)
    to_ll = lambda g: gpd.GeoSeries([g], crs=utm).to_crs(4326).iloc[0]
    return to_ll(poly), to_ll(buf)


def write_geojson(geom, path):
    fc = {'type': 'FeatureCollection',
          'features': [{'type': 'Feature', 'properties': {}, 'geometry': mapping(geom)}]}
    with open(path, 'w') as f:
        json.dump(fc, f)


def build_edges(osm_xml, poly_ll):
    """Every component of the walking network inside the area boundary."""
    G = ox.graph_from_xml(osm_xml, bidirectional=True, simplify=False, retain_all=True)
    G = ox.truncate.truncate_graph_polygon(G, poly_ll, truncate_by_edge=True)
    G = ox.simplify_graph(G)
    G = ox.project_graph(G)
    _, edges = ox.graph_to_gdfs(G)
    return edges[['length', 'geometry']].reset_index(drop=True)


def log(row):
    pd.DataFrame([row]).to_csv(LOG_FILE, mode='a',
                               header=not os.path.exists(LOG_FILE), index=False)

In [ ]:
metro_files = sorted(f for f in os.listdir(INPUT_DIR)
                     if f.startswith('metro_') and f.endswith('.gpkg'))
assert len(metro_files) == 92, f'{len(metro_files)} boundary files, expected 92'

failed = []
t_start = time.time()
for idx, file in enumerate(metro_files, 1):
    metro_name = file[:-5]
    out_drive = f'{OUTPUT_DIR}/{metro_name}_pedestrian.gpkg'
    if os.path.exists(out_drive):
        print(f'[{idx:>2}/92] {metro_name:<34s} already written')
        continue

    t0 = time.time()
    osm_xml = f'{WORK_DIR}/{metro_name}.osm'
    geojson = f'{WORK_DIR}/{metro_name}.geojson'
    out_local = f'{WORK_DIR}/{metro_name}_pedestrian.gpkg'
    status, n_edges, err, comp = 'ok', 0, '', {}
    try:
        poly_ll, buf_ll = metro_polygons(f'{INPUT_DIR}/{file}')
        write_geojson(buf_ll, geojson)
        run(['osmium', 'extract', '-p', geojson, '-s', 'complete_ways',
             WALK_PBF, '-o', osm_xml, '--overwrite'])
        edges = build_edges(osm_xml, poly_ll)
        edges.to_file(out_local, driver='GPKG')
        shutil.copy(out_local, out_drive + '.part')
        os.replace(out_drive + '.part', out_drive)
        n_edges = len(edges)
        comp = mx.component_summary(edges)
        print(f"[{idx:>2}/92] {metro_name:<34s} {n_edges:>8,} edges  "
              f"{comp['n_components']:>5,} components  "
              f"{comp['km_outside_largest']:>7,.0f} km outside the largest  "
              f"{(time.time() - t0) / 60:.1f} min")
    except Exception as e:
        status, err = 'failed', f'{type(e).__name__}: {e}'
        failed.append(metro_name)
        print(f'[{idx:>2}/92] {metro_name:<34s} ERROR: {err}')
        traceback.print_exc()
    finally:
        for f in (osm_xml, geojson, out_local):
            if os.path.exists(f):
                os.remove(f)
    log({'metro': metro_name, 'status': status, 'n_edges': n_edges,
         'minutes': round((time.time() - t0) / 60, 2), 'error': err[:300],
         'time': time.ctime(), **comp})

print(f'\nfinished in {(time.time() - t_start) / 60:.1f} min, failed: {len(failed)}')
for m in failed:
    print('  -', m)

## 4. Summary

In [ ]:
done = (pd.read_csv(LOG_FILE).drop_duplicates('metro', keep='last')
          .query('status == "ok"'))
written = sorted(f for f in os.listdir(OUTPUT_DIR) if f.endswith('_pedestrian.gpkg'))
print(f'{len(written)} network files in {OUTPUT_DIR}')
assert len(written) == 92, 'run the build cell again to complete the missing areas'
if 'n_components' in done.columns:
    print(f"components per area   median {done['n_components'].median():.0f}, "
          f"max {done['n_components'].max():.0f}")
    print(f"length outside the largest component, all areas: "
          f"{done['km_outside_largest'].sum():,.0f} km")